# DMPC Deep Dive — ISR-RL-DMPC

In-depth walkthrough of the DMPC + ADMM pipeline and the ISR support modules.

**Sections**
1. [Formation Geometry](#1) — all six formation types visualised
2. [ADMM Convergence Analysis](#2) — effect of ρ, noise, and drone count
3. [DMPC Multi-Step Simulation](#3) — tracking a moving reference
4. [ISR Modules Tour](#4) — mission planner, threat assessor, task allocator

**Prerequisites:** `pip install -e .` and `pip install -r requirements/dev.txt`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11})
COLORS = ['steelblue', 'tomato', 'seagreen', 'goldenrod', 'mediumpurple', 'darkorange']
print('Imports OK')

<a id='1'></a>
## 1 · Formation Geometry

`FormationGeometry` generates desired 3-D positions for all six
formation types: **Line**, **Wedge**, **Column**, **Circular**, **Grid**, **Sphere**.
Each formation is centred at the origin and can be rotated by a heading angle.

In [ ]:
from isr_rl_dmpc import (
    FormationType, FormationConfig, FormationGeometry,
)

N = 6
center = np.zeros(3)
cfg_base = FormationConfig(scale=50.0, spacing=12.0)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, ftype in zip(axes, FormationType):
    cfg = FormationConfig(type=ftype, scale=50.0, spacing=12.0)
    geom = FormationGeometry(cfg)
    positions = geom.generate_desired_positions(N, center, heading=0.0)

    xs = [positions[i][0] for i in range(N)]
    ys = [positions[i][1] for i in range(N)]
    zs = [positions[i][2] for i in range(N)]

    sc = ax.scatter(xs, ys, c=range(N), cmap='tab10', s=140, zorder=5)
    for i in range(N):
        ax.annotate(str(i), (xs[i], ys[i]),
                    textcoords='offset points', xytext=(5, 5), fontsize=9)
    ax.set_aspect('equal')
    ax.set_title(ftype.value.capitalize(), fontweight='bold')
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.grid(alpha=0.3)

    # Z range as subtitle
    z_min, z_max = min(zs), max(zs)
    ax.set_title(f'{ftype.value.capitalize()}  (Z: {z_min:.0f}–{z_max:.0f} m)', fontweight='bold')

plt.suptitle(f'Formation Geometries — {N} Drones', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('formations.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved formations.png')

In [ ]:
# --- Effect of heading rotation on Wedge formation --------------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
geom_w = FormationGeometry(FormationConfig(type=FormationType.WEDGE, scale=40.0, spacing=10.0))

for ax, heading_deg in zip(axes, [0, 45, 90, 135]):
    heading = np.deg2rad(heading_deg)
    pos = geom_w.generate_desired_positions(5, center, heading=heading)
    xs = [pos[i][0] for i in range(5)]
    ys = [pos[i][1] for i in range(5)]
    ax.scatter(xs, ys, c=range(5), cmap='cool', s=120, zorder=5)
    for i in range(5):
        ax.annotate(str(i), (xs[i], ys[i]), textcoords='offset points',
                    xytext=(4, 4), fontsize=9)
    lim = 55
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(f'Wedge — heading {heading_deg}°', fontweight='bold', fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Wedge Formation under Heading Rotation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='2'></a>
## 2 · ADMM Convergence Analysis

The ADMM penalty parameter **ρ** controls the convergence speed vs.
accuracy trade-off.  Higher ρ gives faster convergence but can overshoot.
This section sweeps ρ and compares convergence curves.

In [ ]:
from isr_rl_dmpc import ADMMConsensus, ADMMConfig

rng = np.random.default_rng(0)

N_DRONES  = 4
DIM       = 3
N_OUTER   = 30
RHO_VALS  = [0.1, 0.5, 1.0, 5.0, 20.0]

true_v     = np.array([50.0, 30.0, 20.0])
proposals  = true_v + rng.uniform(-15, 15, size=(N_DRONES, DIM))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for rho, color in zip(RHO_VALS, COLORS):
    admm = ADMMConsensus(
        num_drones=N_DRONES, dim=DIM,
        config=ADMMConfig(rho=rho, max_iters=1)
    )
    primal_hist, error_hist = [], []
    for _ in range(N_OUTER):
        v = admm.step(proposals)
        primal_hist.append(np.linalg.norm(admm.get_primal_residuals()))
        error_hist.append(np.linalg.norm(v - true_v))

    iters = np.arange(1, N_OUTER + 1)
    ax1.semilogy(iters, primal_hist, lw=2, color=color, label=f'ρ={rho}')
    ax2.semilogy(iters, error_hist,  lw=2, color=color, label=f'ρ={rho}')

for ax, title, ylabel in [
    (ax1, 'Primal Residual ‖z − v‖', 'Primal residual'),
    (ax2, 'Consensus Error ‖v − v*‖', 'Error [m]'),
]:
    ax.axhline(1e-3, color='grey', ls='--', lw=1.2, label='tol = 1e-3')
    ax.set_xlabel('ADMM iteration')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('ADMM Convergence — Effect of ρ', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('admm_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved admm_convergence.png')

In [ ]:
# --- Effect of swarm size on ADMM convergence speed -------------------------
fig, ax = plt.subplots(figsize=(10, 5))
rho = 1.0

for n, color in zip([2, 4, 8, 16], COLORS):
    props_n = true_v + rng.uniform(-15, 15, size=(n, DIM))
    admm_n  = ADMMConsensus(
        num_drones=n, dim=DIM,
        config=ADMMConfig(rho=rho, max_iters=1)
    )
    errors = []
    for _ in range(N_OUTER):
        v_n = admm_n.step(props_n)
        errors.append(np.linalg.norm(v_n - true_v))
    ax.semilogy(range(1, N_OUTER + 1), errors, lw=2, color=color, label=f'{n} drones')

ax.axhline(1e-3, color='grey', ls='--', lw=1.2, label='tol = 1e-3')
ax.set_xlabel('ADMM iteration')
ax.set_ylabel('Consensus error ‖v − v*‖ [m]')
ax.set_title(f'ADMM Consensus Error — Swarm Size (ρ={rho})', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<a id='3'></a>
## 3 · DMPC Multi-Step Simulation

Here we simulate **Drone 0** tracking a sinusoidal reference trajectory
while Drones 1–3 maintain fixed hover positions (as static obstacles).

The DMPC solves the QP at each 20 ms step and the plots show:
- Tracking performance (position error over time)
- Solve time per step
- Optimal control inputs

In [ ]:
import time
from isr_rl_dmpc import DMPC, DMPCConfig

N = 4
dt = 0.02
T = 100  # steps (2 seconds)

cfg = DMPCConfig(
    horizon=10,   # shorter horizon for speed
    dt=dt,
    accel_max=8.0,
    collision_radius=3.0,
    Q_base=np.eye(11) * 2.0,
    R_base=np.eye(3) * 0.1,
)
controller = DMPC(num_drones=N, config=cfg)

# Initial states: [p(3), v(3), a(3), yaw, yaw_rate]
init_positions = np.array([
    [0.0,  0.0, 30.0],
    [30.0, 0.0, 30.0],
    [0.0, 30.0, 30.0],
    [30.0,30.0, 30.0],
])
states = [np.zeros(11) for _ in range(N)]
for i in range(N):
    states[i][:3] = init_positions[i]

# Drone 0 tracks a sinusoidal trajectory; others hover
def sinusoidal_ref(step: int, origin: np.ndarray, horizon: int, dt: float) -> np.ndarray:
    ref = np.zeros((11, horizon + 1))
    for k in range(horizon + 1):
        t = (step + k) * dt
        ref[0, k] = origin[0] + 10.0 * np.sin(2 * np.pi * 0.5 * t)
        ref[1, k] = origin[1] + 10.0 * np.cos(2 * np.pi * 0.5 * t)
        ref[2, k] = origin[2]
    return ref

def hover_ref(x: np.ndarray, horizon: int) -> np.ndarray:
    ref = np.zeros((11, horizon + 1))
    ref[:3, :] = x[:3, np.newaxis]
    return ref

# --- Simulation loop --------------------------------------------------------
traj_0   = [states[0][:3].copy()]
ref_0    = []
errors   = []
controls = []
solve_ms = []

for step in range(T):
    refs = [sinusoidal_ref(step, init_positions[0], cfg.horizon, dt)]
    for i in range(1, N):
        refs.append(hover_ref(states[i], cfg.horizon))

    ref_0.append(refs[0][:3, 0].copy())

    t0 = time.perf_counter()
    results = controller(states, refs)
    solve_ms.append((time.perf_counter() - t0) * 1e3)

    for i, res in enumerate(results):
        u = res.get('u', np.zeros(3))
        if i == 0:
            controls.append(u.copy())
        states[i][6:9] += u * dt
        states[i][3:6] += states[i][6:9] * dt
        states[i][:3]  += states[i][3:6] * dt

    traj_0.append(states[0][:3].copy())
    errors.append(np.linalg.norm(states[0][:3] - refs[0][:3, 0]))

traj_0   = np.array(traj_0)   # (T+1, 3)
ref_0    = np.array(ref_0)    # (T, 3)
controls = np.array(controls) # (T, 3)
t_ax     = np.arange(T + 1) * dt

print(f'Mean solve time: {np.mean(solve_ms):.2f} ms  |  Max: {np.max(solve_ms):.2f} ms')
print(f'Mean tracking error: {np.mean(errors):.3f} m')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# XY trajectory
axes[0,0].plot(ref_0[:, 0], ref_0[:, 1], 'k--', lw=1.5, label='Reference', alpha=0.6)
axes[0,0].plot(traj_0[1:, 0], traj_0[1:, 1], color='steelblue', lw=2, label='Drone 0')
for i in range(1, N):
    axes[0,0].scatter(init_positions[i, 0], init_positions[i, 1],
                      s=120, color=COLORS[i], zorder=5, label=f'Drone {i} (hover)')
axes[0,0].set_xlabel('X [m]')
axes[0,0].set_ylabel('Y [m]')
axes[0,0].set_title('XY Trajectory — Sinusoidal Tracking', fontweight='bold')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(alpha=0.3)

# Tracking error over time
axes[0,1].plot(t_ax[1:], errors, color='tomato', lw=2)
axes[0,1].set_xlabel('Time [s]')
axes[0,1].set_ylabel('Position error [m]')
axes[0,1].set_title('Tracking Error over Time', fontweight='bold')
axes[0,1].grid(alpha=0.3)

# Control inputs
for dim, label, color in zip(range(3), ['aₓ', 'aᵧ', 'a_z'], COLORS):
    axes[1,0].plot(t_ax[1:], controls[:, dim], lw=1.5, color=color, label=label)
axes[1,0].axhline(cfg.accel_max,  color='grey', ls='--', lw=1, label=f'±{cfg.accel_max} m/s²')
axes[1,0].axhline(-cfg.accel_max, color='grey', ls='--', lw=1)
axes[1,0].set_xlabel('Time [s]')
axes[1,0].set_ylabel('Acceleration [m/s²]')
axes[1,0].set_title('Optimal Control Inputs (Drone 0)', fontweight='bold')
axes[1,0].legend(fontsize=9)
axes[1,0].grid(alpha=0.3)

# Solve time
axes[1,1].plot(t_ax[1:], solve_ms, color='mediumpurple', lw=1.5)
axes[1,1].axhline(20, color='grey', ls='--', lw=1.5, label='50 Hz deadline (20 ms)')
axes[1,1].set_xlabel('Time [s]')
axes[1,1].set_ylabel('Solve time [ms]')
axes[1,1].set_title('DMPC Solve Time per Step', fontweight='bold')
axes[1,1].legend(fontsize=9)
axes[1,1].grid(alpha=0.3)

plt.suptitle('DMPC Multi-Step Simulation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dmpc_multistep.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved dmpc_multistep.png')

<a id='4'></a>
## 4 · ISR Modules Tour

Quick demonstration of the six ISR support modules.
Each module runs standalone — no simulator required.

In [ ]:
# ── Module 1: Mission Planner ─────────────────────────────────────────────
from isr_rl_dmpc import MissionPlanner, GridDecomposer, WaypointGenerator

planner = MissionPlanner()
decomposer = GridDecomposer(cell_size=50.0)

# Decompose a 400×400 m search area
area_bounds = np.array([[-200, -200, 0], [200, 200, 0]], dtype=float)
cells = decomposer.decompose(area_bounds[0], area_bounds[1])
print(f'Mission Planner: area 400×400 m → {len(cells)} grid cells (50 m each)')

In [ ]:
# ── Module 5: Threat Assessor ─────────────────────────────────────────────
from isr_rl_dmpc import (
    ThreatAssessor, ThreatParameters, ThreatLevel,
    TargetState,
)

assessor = ThreatAssessor(ThreatParameters())

targets = [
    TargetState(target_id=0, position=np.array([80.0, 0.0, 0.0]),
                velocity=np.array([5.0, 2.0, 0.0])),
    TargetState(target_id=1, position=np.array([300.0, 200.0, 0.0]),
                velocity=np.zeros(3)),
    TargetState(target_id=2, position=np.array([50.0, 50.0, 10.0]),
                velocity=np.array([12.0, 8.0, 1.0])),  # fast mover
]

swarm_center = np.zeros(3)

print('Threat Assessment results:')
print(f'  {"Target":<10} {"Threat Level":<18} {"Score":>8}')
print('  ' + '-' * 38)
for tgt in targets:
    assessment = assessor.assess(tgt, swarm_center)
    print(f'  Target {tgt.target_id:<4} {assessment.threat_level.name:<18} {assessment.threat_score:>8.3f}')

In [ ]:
# ── Module 6: Task Allocator (Hungarian Algorithm) ────────────────────────
from isr_rl_dmpc import (
    TaskAllocator, ISRTask, TaskType, TaskStatus, DroneCapability,
    DroneState,
)

N_D = 4
drone_states = [
    DroneState(drone_id=i, position=init_positions[i % 4], battery_level=1.0 - 0.1 * i)
    for i in range(N_D)
]

tasks = [
    ISRTask(task_id=0, task_type=TaskType.SURVEILLANCE,
            position=np.array([100.0, 0.0, 0.0]),   priority=3),
    ISRTask(task_id=1, task_type=TaskType.TRACKING,
            position=np.array([0.0, 100.0, 0.0]),   priority=5),
    ISRTask(task_id=2, task_type=TaskType.INVESTIGATION,
            position=np.array([60.0, 60.0, 0.0]),   priority=4),
]

allocator = TaskAllocator()
assignments = allocator.assign(drone_states, tasks)

print('Task Allocator (Hungarian Algorithm) results:')
print(f'  {"Drone":<8} {"Task":<12} {"Type":<18} {"Priority":>8}')
print('  ' + '-' * 50)
for drone_id, task_id in assignments.items():
    task = next(t for t in tasks if t.task_id == task_id)
    print(f'  Drone {drone_id:<4} Task {task_id:<7} {task.task_type.name:<18} {task.priority:>8}')

In [ ]:
# ── Visualise the task allocation ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

drone_colors = ['steelblue', 'tomato', 'seagreen', 'goldenrod']
task_markers = {'SURVEILLANCE': 's', 'TRACKING': 'D', 'INVESTIGATION': '^'}

for i, ds in enumerate(drone_states):
    ax.scatter(*ds.position[:2], s=160, color=drone_colors[i],
               zorder=5, label=f'Drone {i} (batt {ds.battery_level:.0%})')

task_type_colors = {'SURVEILLANCE': 'navy', 'TRACKING': 'darkred', 'INVESTIGATION': 'darkgreen'}
for task in tasks:
    t_name = task.task_type.name
    ax.scatter(*task.position[:2],
               s=200, marker=task_markers[t_name],
               color=task_type_colors[t_name], zorder=5,
               label=f'Task {task.task_id} — {t_name} (p={task.priority})')

# Draw assignment arrows
for drone_id, task_id in assignments.items():
    d_pos = drone_states[drone_id].position[:2]
    t_pos = next(t for t in tasks if t.task_id == task_id).position[:2]
    ax.annotate('', xy=t_pos, xytext=d_pos,
                arrowprops=dict(arrowstyle='->', lw=1.8,
                                color=drone_colors[drone_id], alpha=0.7))

ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_title('Task Allocation — Hungarian Algorithm', fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('task_allocation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task_allocation.png')

## Summary

| Module | Key Classes | Demonstrated |
| :--- | :--- | :--- |
| Formation Controller | `FormationGeometry`, `FormationType` | §1 — all 6 formation types |
| ADMM Consensus | `ADMMConsensus`, `ADMMConfig` | §2 — ρ sweep & swarm size |
| DMPC Controller | `DMPC`, `DMPCConfig` | §3 — sinusoidal tracking |
| Mission Planner | `MissionPlanner`, `GridDecomposer` | §4 — grid decomposition |
| Threat Assessor | `ThreatAssessor`, `ThreatLevel` | §4 — threat scoring |
| Task Allocator | `TaskAllocator`, `ISRTask` | §4 — Hungarian assignment |

**Continue with:**
- `03_training_curves.ipynb` — MAPPO training metrics
- `04_mission_analysis.ipynb` — per-scenario results
- `05_comparison_analysis.ipynb` — Pure DMPC vs DMPC-RL